# 🧚 동화 AI — 풀학습 노트북 (세션 분할 버전)

## 사용법
1. **매 세션마다 이 노트북을 처음부터 실행하세요**
2. 자동으로 이전 체크포인트에서 이어서 학습해요
3. 약 3시간마다 Colab 세션 끊기면 다시 실행하면 돼요

| 세팅 | 값 |
|------|----|
| 모델 | EXAONE-3.5-7.8B (A100) |
| 총 학습 데이터 | 100,000개 |
| 세션당 스텝 | 500스텝 (~3시간) |
| 총 스텝 | ~9,375 |

> 💡 **런타임 → A100 GPU** 선택 후 시작하세요!

## ✅ 1단계 — GPU 확인 & 라이브러리 설치

In [ ]:
!nvidia-smi
import torch
vram = torch.cuda.get_device_properties(0).total_memory/1024**3
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {vram:.1f} GB')

In [ ]:
!pip install -q bitsandbytes>=0.45.0
!pip install -q transformers>=4.40.0 peft>=0.10.0 datasets accelerate sentencepiece protobuf
print('✅ 설치 완료!')

## ✅ 2단계 — Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/fairytale_ai'
CKPT_DIR = f'{WORK_DIR}/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(f'{WORK_DIR}/final_model', exist_ok=True)

# 현재 세션 번호 확인
ckpts = sorted([d for d in os.listdir(CKPT_DIR) if d.startswith('checkpoint-')]) if os.path.exists(CKPT_DIR) else []
last_ckpt = os.path.join(CKPT_DIR, ckpts[-1]) if ckpts else None
last_step = int(ckpts[-1].split('-')[-1]) if ckpts else 0

print(f'✅ Drive 마운트 완료')
if last_ckpt:
    print(f'🔄 이전 체크포인트 발견: {ckpts[-1]} (스텝 {last_step})')
    print(f'   → 이어서 학습합니다')
else:
    print(f'🆕 첫 번째 세션 — 처음부터 시작합니다')

## ✅ 3단계 — 데이터 로드

In [ ]:
import json, random
from datasets import Dataset

# Drive → 로컬 복사
print('데이터 복사 중...')
!cp '/content/drive/MyDrive/fairytale_ai/train.jsonl' /content/train.jsonl
!cp '/content/drive/MyDrive/fairytale_ai/val.jsonl'   /content/val.jsonl

TRAIN_SAMPLES = 100000
VAL_SAMPLES   = 3000

def load_jsonl(path, n=None):
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    if n and len(rows) > n:
        random.seed(42)
        rows = random.sample(rows, n)
    return Dataset.from_list(rows)

train_ds = load_jsonl('/content/train.jsonl', TRAIN_SAMPLES)
val_ds   = load_jsonl('/content/val.jsonl',   VAL_SAMPLES)
print(f'✅ 학습: {len(train_ds):,}개 / 검증: {len(val_ds):,}개')

## ✅ 4단계 — 모델 & LoRA 설정

In [ ]:
import os, torch, gc
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
gc.collect()
torch.cuda.empty_cache()

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
import transformers.modeling_utils

vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
if vram >= 35:
    MODEL_NAME = 'LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct'
elif vram >= 20:
    MODEL_NAME = 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct'
else:
    MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

print(f'VRAM: {vram:.1f}GB → 모델: {MODEL_NAME}')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print('\n모델 로드 중...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False

# ✅ EXAONE PEFT 호환 패치 (transformer.wte 사용)
if "EXAONE" in MODEL_NAME:
    if hasattr(model, "transformer") and hasattr(model.transformer, "wte"):
        model.get_input_embeddings = lambda *args, **kwargs: model.transformer.wte
        model.set_input_embeddings = lambda value, *args, **kwargs: setattr(model.transformer, 'wte', value)
    if hasattr(model, "lm_head"):
        model.get_output_embeddings = lambda *args, **kwargs: model.lm_head
        model.set_output_embeddings = lambda value, *args, **kwargs: setattr(model, 'lm_head', value)

# ✅ 어텐션 인터페이스 패치
if hasattr(transformers.modeling_utils, "ALL_ATTENTION_FUNCTIONS"):
    funcs = transformers.modeling_utils.ALL_ATTENTION_FUNCTIONS
    if not hasattr(funcs.__class__, "get_interface"):
        def _get_interface(self, attn_implementation, fallback):
            if hasattr(self, "get"): return self.get(attn_implementation, fallback)
            if hasattr(self, attn_implementation): return getattr(self, attn_implementation)
            return fallback
        funcs.__class__.get_interface = _get_interface

model = prepare_model_for_kbit_training(model)
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none',
)
model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f'\n✅ 준비 완료!')
print(f'   모델: {MODEL_NAME}')
print(f'   학습 파라미터: {trainable:,} ({100*trainable/total:.2f}%)')
print(f'   VRAM: {torch.cuda.memory_allocated(0)/1024**3:.1f} / {vram:.1f} GB')

## ✅ 5단계 — 학습 (자동 이어하기)

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

MAX_LEN = 1024

def tokenize_fn(examples):
    out = tokenizer(examples['text'], truncation=True, max_length=MAX_LEN, padding='max_length')
    out['labels'] = out['input_ids'].copy()
    return out

print('토크나이즈 중...')
train_tok = train_ds.map(tokenize_fn, batched=True, remove_columns=['text'], num_proc=2)
val_tok   = val_ds.map(tokenize_fn,   batched=True, remove_columns=['text'], num_proc=2)
print(f'✅ 완료!')

In [ ]:
# ── 세션 설정 ──
STEPS_PER_SESSION = 500   # 세션당 스텝 수 (~3시간)
TOTAL_STEPS       = 9375  # 전체 스텝 (100,000샘플 / 배치32 * 3에폭)

next_step = last_step + STEPS_PER_SESSION
is_done   = last_step >= TOTAL_STEPS

print(f'📊 학습 진행 현황')
print(f'   완료 스텝: {last_step:,} / {TOTAL_STEPS:,} ({100*last_step/TOTAL_STEPS:.1f}%)')
print(f'   이번 세션: {last_step:,} → {min(next_step, TOTAL_STEPS):,}')

if is_done:
    print('\n🎉 학습이 이미 완료되었습니다! 6단계(저장)로 넘어가세요.')
else:
    args = TrainingArguments(
        output_dir=CKPT_DIR,
        max_steps=min(next_step, TOTAL_STEPS),
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        lr_scheduler_type='cosine',
        fp16=True,
        optim='paged_adamw_8bit',
        logging_steps=20,
        eval_strategy='steps',
        eval_steps=200,
        save_strategy='steps',
        save_steps=100,
        save_total_limit=3,
        load_best_model_at_end=False,
        report_to='none',
        dataloader_num_workers=0,
        remove_unused_columns=True,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    )

    print(f'\n🚀 학습 시작! (이번 세션 {STEPS_PER_SESSION}스텝)')
    trainer.train(resume_from_checkpoint=last_ckpt)
    print(f'\n✅ 이번 세션 완료! 다음 세션에서 이어하려면 노트북을 다시 실행하세요.')
    print(f'   남은 스텝: {max(0, TOTAL_STEPS - min(next_step, TOTAL_STEPS)):,}')

## ✅ 6단계 — 최종 저장 & 테스트 (학습 완료 후 실행)

In [ ]:
# 학습이 완전히 끝난 후 실행하세요
FINAL_DIR = f'{WORK_DIR}/final_model'
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

size_mb = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, files in os.walk(FINAL_DIR) for f in files
) / 1024**2
print(f'✅ 최종 모델 저장 완료! ({size_mb:.0f} MB)')
print(f'   위치: {FINAL_DIR}')

In [ ]:
import torch

def generate(prompt, max_new_tokens=400):
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.15,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

tests = [
    ('<s>[INST] 장르: 판타지, 대상 연령: 유아\n작은 토끼가 마법의 당근을 발견하는 이야기를 써줘 [/INST]', '판타지'),
    ('<s>[INST] 장르: 사회관계, 대상 연령: 초등_저학년\n친구와 싸운 아이가 화해하는 이야기를 써줘 [/INST]', '사회관계'),
]
for prompt, genre in tests:
    print(f'\n{"="*50}\n📖 [{genre}]\n{"="*50}')
    print(generate(prompt)[:400])

## 📋 세션 관리

| 세션 | 스텝 범위 | 상태 |
|------|-----------|------|
| 1 | 0 → 500 | ⬜ |
| 2 | 500 → 1000 | ⬜ |
| 3 | 1000 → 1500 | ⬜ |
| ... | ... | ... |
| 19 | 9000 → 9375 | ⬜ |

완료한 세션은 ⬜ → ✅ 로 표시하면서 진행하세요!